In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import map_coordinates

import plotly.graph_objects as go

In [ ]:
def get_background_image(url_img):
    """Function to retrieve image by URL.

    Parameters
    ----------
    url_img : str
        URL of the shared image link.

    Output
    ------
    ``numpy.array``
        Output image in array format.
    """
    from PIL import Image
    import requests
    from io import BytesIO

    response = requests.get(url_img)
    img = Image.open(BytesIO(response.content))
    return np.asarray(img)  # RGBA

In [ ]:
# Upload image url

img_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRq3-x1HjeAX7y8UkTLkOhP_YyEbziDK5jy4HIgP-7x5SH7aqHkaRwm80PN&s=10"
pixels = get_background_image(img_url)

In [ ]:
class LensImage:

    G = 6.67430e-11 # m^3 kg^(-1) s^(-2)
    c = 299_792_458 # m/s
    solar_mass = 1.989e30 # kg
    light_year = 9.461e15 # m

    def __init__(
        self,
        input_image,
        number_of_solar_masses,
        dist,
        FOV_mas
    ):
        self.input_image = input_image
        self.number_of_solar_masses = number_of_solar_masses
        self.dist = dist
        self.FOV_mas = FOV_mas

        self.define_variables()
        self.adjust_field_of_view()
        self.create_canvas()
        self.compute_distances()

    def define_variables(self):

        self.M = self.number_of_solar_masses * self.solar_mass
        self.D = self.dist * self.light_year
        self.pixels_3d = np.atleast_3d(self.input_image)
        self.height, self.width, self.n_col = self.pixels_3d.shape

        self.FOV_radians = (self.FOV_mas * np.pi / (180 * 3600 * 1000))
        self.s = self.FOV_radians / self.width
        self.b_critical = (3 * np.sqrt(3) * self.G * self.M / self.c**2)

    def adjust_field_of_view(self):

        max_photon_ring_radius = 0.45 * min(self.width, self.height)
        minimum_s = (self.b_critical / (self.D * max_photon_ring_radius))
        if self.s < minimum_s:
            self.s = minimum_s
            self.FOV_radians = self.s * self.width
        self.FOV_mas = (self.FOV_radians * (180 * 3600 * 1000) / np.pi)
        self.photon_ring_radius_pixels = (self.b_critical / (self.D * self.s))

    def create_canvas(self):

        self.canvas_height = 2 * self.height
        self.canvas_width = 2 * self.width

        self.idcs = np.meshgrid(np.arange(self.canvas_width), np.arange(self.canvas_height))

        self.canvas_xc = self.width
        self.canvas_yc = self.height

    def compute_distances(self):

        self.dx = self.idcs[0] - self.canvas_xc
        self.dy = self.idcs[1] - self.canvas_yc

        self.distance = np.sqrt(self.dx**2 + self.dy**2)

        self.b = self.D * self.distance * self.s
        self.valid = self.distance > 0
        self.alpha = np.zeros_like(self.distance)
        self.alpha[self.valid] = (4 * self.G * self.M / (self.c**2 * self.b[self.valid]))
        self.pixel_shift = self.alpha / self.s
        self.inside_shadow = self.b <= self.b_critical

        self.radial_x = np.zeros_like(self.distance)
        self.radial_y = np.zeros_like(self.distance)
        self.radial_x[self.valid] = (self.dx[self.valid] / self.distance[self.valid])
        self.radial_y[self.valid] = (self.dy[self.valid] / self.distance[self.valid])
        self.source_x = (self.idcs[0] - self.pixel_shift * self.radial_x)
        self.source_y = (self.idcs[1] - self.pixel_shift * self.radial_y)

    def create_image(self, xc, yc):

        x_start = self.canvas_xc - xc
        y_start = self.canvas_yc - yc

        x_end = x_start + self.width
        y_end = y_start + self.height

        source_x = (self.source_x[y_start:y_end, x_start:x_end] - x_start)
        source_y = (self.source_y[y_start:y_end, x_start:x_end] - y_start)
        shadow = self.inside_shadow[y_start:y_end, x_start:x_end]
        lensed_pixels = np.zeros_like(self.pixels_3d)

        for color in range(self.n_col):

            lensed_pixels[:, :, color] = map_coordinates(
                self.pixels_3d[:, :, color],
                [source_y, source_x],
                order=1,
                mode="constant",
                cval=0
            )
        lensed_pixels[shadow] = 0
        return lensed_pixels


In [ ]:
lens = LensImage(
    input_image=pixels,
    number_of_solar_masses=1000000,
    dist=300000,
    FOV_mas=100000
)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

xc_slider = widgets.IntSlider(
    value=lens.width // 2,
    min=0,
    max=lens.width - 1,
    step=1,
    description="Center X:",
    continuous_update=False
)

yc_slider = widgets.IntSlider(
    value=lens.height // 2,
    min=0,
    max=lens.height - 1,
    step=1,
    description="Center Y:",
    continuous_update=False
)

mass_slider = widgets.SelectionSlider(
    options=[100000, 300000, 500000, 1000000, 3000000],
    value=500000,
    description="Mass:",
    continuous_update=False
)

distance_slider = widgets.SelectionSlider(
    options=[100000, 150000, 200000, 250000, 300000],
    value=200000,
    description="Distance:",
    continuous_update=False
)

fov_slider = widgets.SelectionSlider(
    options=[1000, 2000, 5000, 10000, 20000],
    value=5000,
    description="FOV:",
    continuous_update=False
)

In [ ]:
current_lens = None
current_settings = None

def update_image(xc, yc, mass, distance, fov):
    global current_lens, current_settings

    new_settings = (mass, distance, fov)

    # Recompute the distortion only if mass, distance, or FOV changes
    if new_settings != current_settings:
        current_lens = LensImage(
            input_image=pixels,
            number_of_solar_masses=mass,
            dist=distance,
            FOV_mas=fov
        )
        current_settings = new_settings

    # Moving only xc or yc uses the already-computed canvas
    new_image = current_lens.create_image(xc, yc)

    plt.figure(figsize=(7, 6))
    plt.imshow(new_image)
    plt.axis("off")
    plt.show()

In [ ]:
controls = widgets.VBox([
    xc_slider,
    yc_slider,
    mass_slider,
    distance_slider,
    fov_slider
])

output = widgets.interactive_output(
    update_image,
    {
        "xc": xc_slider,
        "yc": yc_slider,
        "mass": mass_slider,
        "distance": distance_slider,
        "fov": fov_slider
    }
)

display(controls, output)

Output()